# Ge'ez Morphology: ByT5 Fine-Tuning on Google Colab / Kaggle

This notebook fine-tunes a subword-free **ByT5** model on Classical Ge'ez root-and-pattern morphology.
Run this with a **GPU runtime** (T4 GPU is free on Colab & Kaggle).

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets accelerate torch

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

### 2. Clone Repositories or Load Dataset

In [ ]:
# Clone repositories directly from GitHub
!git clone https://github.com/esubaalew/geez-morphology.git
!git clone https://github.com/esubaalew/geez-dataset.git

# Ensure output directories exist
!mkdir -p ./checkpoints

### 3. Load & Preprocess Morphological Training Pairs

In [ ]:
import json
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

MODEL_NAME = "google/byt5-small"
DATASET_PATH = "geez-dataset/output/synthetic_morphology_train.jsonl"
SAMPLE_LIMIT = 30000  # Adjust as needed (up to 145,000)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

samples = []
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= SAMPLE_LIMIT:
            break
        d = json.loads(line)
        feat = d.get("features", {})
        surface = d.get("surface_form", "")
        
        input_text = f"analyze: {surface}"
        target_text = f"root: {feat.get('root', '')} | tense: {feat.get('tense', '')} | person: {feat.get('person', '')} | gender: {feat.get('gender', '')} | number: {feat.get('number', '')} | stem: {feat.get('stem', '')}"
        samples.append({"input_text": input_text, "target_text": target_text})

print(f"Loaded {len(samples)} training pairs.")
print(f"Example input:  {samples[0]['input_text']}")
print(f"Example target: {samples[0]['target_text']}")

### 4. Dataset Class & Train/Eval Split

In [ ]:
class GeezDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=96):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        inputs = self.tokenizer(item["input_text"], max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        targets = self.tokenizer(item["target_text"], max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt")
        labels = targets["input_ids"].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": labels
        }

split = int(len(samples) * 0.9)
train_ds = GeezDataset(samples[:split], tokenizer)
eval_ds = GeezDataset(samples[split:], tokenizer)
print(f"Train size: {len(train_ds)}, Eval size: {len(eval_ds)}")

### 5. Training with GPU Acceleration

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-4,
    num_train_epochs=3,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model)
)

trainer.train()

### 6. Interactive Model Test

In [ ]:
def analyze_word(word: str):
    input_text = f"analyze: {word}"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=96)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test forms
for test_word in ["ይቀትሉ", "ተቀተለት", "ይባርኩ", "ይቀድሱ"]:
    print(f"{test_word} -> {analyze_word(test_word)}")